# 02 · Merging mentions into unique risks

**Step 2 of the pipeline.** Step 1 returns each principal risk once per page it appears on. A report mentions the
same risk in several places, at different depth: a row in a summary table, a full section with mitigations
elsewhere. One call per report groups those mentions and writes one final record per unique risk: title, category,
a 2-3 sentence description, the mitigation, and every location. Evaluating this step is notebook 02b.

- **Why a separate step:** finding and copying (step 1) is a different job from de-duplicating and summarising.
  Each can be evaluated and debugged on its own.
- **Why one call per report:** deciding whether two mentions are the same risk needs every mention in view.
- **Why locations are copied in code:** the model picks which mentions belong together; page and section are taken
  from those mentions, so they are always what step 1 actually extracted.

The code lives in `annual_report_reader/merge/`, one module per section (named in its heading). It was written
here first and then lifted. This notebook shows each stage working on a real extraction run; the modules hold the
definitions and are unit-tested (`tests/merge/`).

## 1. Config → `merge/config.py`

Everything that decides what a merge produces lives in one frozen object, saved with the run. The input is an
extraction run, named by id: there is no "latest", because the newest run is often an experiment (another model,
another prompt), and a merge must never build on one by accident.
Paths are relative to the repo root. Secrets stay in `.env`.

In [1]:
from dotenv import load_dotenv

from annual_report_reader.extraction.config import REPO_ROOT
from annual_report_reader.merge.config import MergeConfig

load_dotenv(REPO_ROOT / ".env")  # endpoint, key and deployment name

CONFIG = MergeConfig(extraction_run="extract-20261112T000123Z")  # Luna, high effort, prompt extract
CONFIG

MergeConfig(extraction_run='extract-20261112T000123Z', deployment='gpt-5.6-luna-carl-test', reasoning_effort='medium', prompt_dir=PosixPath('prompts'), prompt='merge', price_in_per_m=0.2, price_out_per_m=1.2)

## 2. Input: the extraction run → `extraction/runs.py`

The merge reads one saved extraction run (notebook 01). A run with a **failed** page is refused: merging a partial
report gives output that looks complete but is not. Re-run notebook 01 instead.

Every configured page is listed with its status, including pages with no risks.

In [2]:
from annual_report_reader.extraction.runs import RUNS_DIR, load_run, risks_per_page

run = load_run(REPO_ROOT / RUNS_DIR / f"{CONFIG.extraction_run}.json")
if run.failed_pages:
    raise ValueError(f"{run.run_id}: extraction failed on pages {run.failed_pages}; re-run notebook 01")

print(f"{run.run_id}: {len(run.risks)} risks, prompt {run.config.prompt}, model {run.config.deployment}")
risks_per_page(run).T

extract-20261112T000123Z: 12 risks, prompt extract, model gpt-5.6-luna-carl-test


page,50,51,71,72,73,74,85,118
risks,0,3,1,0,3,3,1,1
status,ok,ok,ok,ok,ok,ok,ok,ok


## 3. Mentions → `merge/mentions.py`

Each step-1 risk becomes a **mention**: the same fields, plus an id the model uses to say which mentions belong
together. The id is the page and the position on that page in the run (`p51-0`, `p51-1`, ...): readable, and the
same every time the same run is loaded.

`confidence` is left out: the merge prompt does not use it, and carrying it into the final record is a later
decision (STRETCH).

In [3]:
import pandas as pd

from annual_report_reader.merge.mentions import mentions_from_run

mentions = mentions_from_run(run)
print(f"{len(mentions)} mentions on pages {sorted({m.page for m in mentions})}")
pd.DataFrame([m.model_dump() for m in mentions])[["mention_id", "category", "secondary_category", "title", "section"]]

12 mentions on pages [51, 71, 73, 74, 85, 118]


,mention_id,category,secondary_category,title,section
0,p51-0,regulatory,market,Geopolitics and regulatory framework,Main risks - Geopolitics and regulatory framework
1,p51-1,operational,supply chain,Project execution,Main risks - Project execution
2,p51-2,cyber,operational,Cyber attacks,Main risks - Cyber attacks
3,p71-0,climate,financial,Carbon taxes and tariffs,E1 Climate Change - Financial risks or opportu...
4,p73-0,financial,operational,Cost implication of injuries in own workforce,S1 Own Workforce - Financial risks or opportun...
5,p73-1,financial,operational,Cost implications of injuries to contractors a...,S2 Workers in the value chain - Financial risk...
6,p73-2,regulatory,supply chain,Fines related to forced or child labour,S2 Workers in the value chain - Financial risk...
7,p74-0,market,financial,Insufficient market conditions,G1 Business Conduct - Financial risks or oppor...
8,p74-1,cyber,operational,Cyber security risks,G1 Business Conduct - Financial risks or oppor...
9,p74-2,regulatory,financial,Risk of corruption and bribery,G1 Business Conduct - Financial risks or oppor...


## 4. Schema → `merge/schema.py`

Two shapes, because the model and the code each do part of the job:

- **What the call returns** (`MergedRisks`): per unique risk, the ids of its mentions (most informative first) and
  the written fields. Sent as strict structured output, like step 1. The model never writes a page or a section.
- **The final record** (`RiskRecord`): the call's fields plus what code adds: an id (`r00`, `r01`, ... in output
  order), the `locations` (page and section of each mention, copied in the model's order), and the `mention_ids`,
  so every record can be traced to the step-1 mentions it was built from.

The fields match the step-2 golden set (`eval/golden/merging_and_summarizing.yaml`), so the output can be evaluated
without conversion. Category is step 1's `Category`.

In [4]:
# The JSON schema the model is constrained to.
import json

from annual_report_reader.merge.schema import MergedRisks

print(json.dumps(MergedRisks.model_json_schema(), indent=1)[:1500], "...")

{
 "$defs": {
  "Category": {
   "enum": [
    "financial",
    "operational",
    "regulatory",
    "market",
    "climate",
    "cyber",
    "supply chain",
    "other"
   ],
   "title": "Category",
   "type": "string"
  },
  "MergedRisk": {
   "additionalProperties": false,
   "description": "One unique risk, as the model returns it.",
   "properties": {
    "mention_ids": {
     "description": "Ids of all mentions of this risk, most informative first",
     "items": {
      "type": "string"
     },
     "title": "Mention Ids",
     "type": "array"
    },
    "title": {
     "description": "Short title, at most about 8 words, sentence case",
     "title": "Title",
     "type": "string"
    },
    "category": {
     "$ref": "#/$defs/Category"
    },
    "secondary_category": {
     "anyOf": [
      {
       "$ref": "#/$defs/Category"
      },
      {
       "type": "null"
      }
     ],
     "description": "A second category that also clearly fits, or null"
    },
    "description":

## 5. Prompt → `merge/prompts.py`

The system prompt is a file, `prompts/<prompt>.md`; every saved run also stores the exact prompt sent.
The mentions go in one user message as a JSON list, each with its id and the fields step 1 extracted.

Unlike step 1, the prompt has no principal-risk definition: step 2 merges and summarises what step 1 found, it does
not filter it again.

In [5]:
# The messages sent (cut short).
from annual_report_reader.merge.prompts import load_merge_prompt, mentions_payload

prompt = load_merge_prompt(CONFIG.resolve(CONFIG.prompt_dir), CONFIG.prompt)
for m in prompt.invoke({"mentions": mentions_payload(mentions)}).to_messages():
    print(f"--- {m.type} ({len(m.content):,} chars)\n{m.content[:700]}\n[...]\n")

--- system (1,948 chars)
You get the principal risks extracted page by page from one company annual report. Each item is a "mention" with an
id. The same risk is often mentioned in several places at different depth, for example a short row in a summary
table and a full section with mitigations elsewhere in the report.

Merge the mentions into a list of unique risks, and write the final record for each.

When mentions are the same risk:

- Same risk means the same source of harm and the same effect on the company. Different wording, depth, section,
  page or category does not matter.
- Related but distinct risks stay separate, for example the same hazard for different affected groups, or two
  different causes listed
[...]

--- human (10,062 chars)
[
 {
  "mention_id": "p51-0",
  "page": 51,
  "title": "Geopolitics and regulatory framework",
  "category": "regulatory",
  "secondary_category": "market",
  "section": "Main risks - Geopolitics and regulatory framework",
  "description": "I

## 6. Chain → `merge/chain.py`

The same setup as step 1: `AzureChatOpenAI` with strict JSON-schema structured output (`MergedRisks`) and
`include_raw=True` for the token usage and any parsing error. The chain takes the list of mentions and builds the
user message itself.

**One call per report.** A failed call (an error after the client's retries, or no parsed output) comes back as a
`MergeCallResult` with an `error`, not as an exception, so the attempt is still saved (section 8).

In [6]:
import time
from datetime import UTC, datetime

from annual_report_reader.chat import azure_chat_model
from annual_report_reader.merge.chain import merge_chain, merge_mentions
from annual_report_reader.merge.runs import merge_run_id_for

chain = merge_chain(prompt, azure_chat_model(CONFIG.deployment, CONFIG.reasoning_effort))

started_at = datetime.now(UTC)
merge_run_id = merge_run_id_for(started_at)
start = time.perf_counter()
call = merge_mentions(chain, mentions, CONFIG, merge_run_id)
elapsed = time.perf_counter() - start

status = f"FAILED: {call.error}" if call.error else f"{len(call.risks)} unique risks"
print(f"{merge_run_id}: {status} | {call.usage} | {elapsed:.0f}s | ${CONFIG.cost(call.usage):.4f}")

merge-20261112T012912Z-53ad: 9 unique risks | input_tokens=2792 output_tokens=1105 reasoning_tokens=258 | 6s | $0.0019


## 7. Records → `merge/records.py`

The model's answer is checked in code before it becomes the final records, because a JSON schema cannot say "every
id must be a real mention, used once":

- an **unknown** id, or one already **used** (by an earlier risk or twice in the same one), is dropped: the first
  use wins, as for the judge's pairs in step 1;
- a risk left with **no valid ids** is dropped;
- a mention the model **left out** becomes its own record, copied as is, so nothing step 1 found is silently lost
  (plan: recall first). Its description is step 1's verbatim text, so it can be longer than 3 sentences.

Every one of these is kept as a `Problem` and saved with the run. `locations` are copied from the mentions, in the
model's order.

In [7]:
from annual_report_reader.merge.records import build_records

records, problems = build_records(call.risks, mentions) if not call.error else ([], [])
n_merged = sum(len(r.mention_ids) > 1 for r in records)
print(f"{len(mentions)} mentions → {len(records)} records ({n_merged} merged from several)")
for p in problems:
    print(f"PROBLEM {p.kind}: {p.mention_id or ''} {p.risk_title!r}")
pd.DataFrame([{**r.model_dump(exclude={"locations"}), "pages": [loc.page for loc in r.locations]} for r in records])[
    ["id", "title", "category", "secondary_category", "pages", "mention_ids"]
]

12 mentions → 9 records (2 merged from several)


,id,title,category,secondary_category,pages,mention_ids
0,r00,Geopolitics and regulatory framework,regulatory,market,[51],[p51-0]
1,r01,Project execution,operational,supply chain,[51],[p51-1]
2,r02,Cyber security risks,cyber,operational,"[51, 118, 74]","[p51-2, p118-0, p74-1]"
3,r03,Carbon taxes and tariffs,financial,climate,"[85, 71]","[p85-0, p71-0]"
4,r04,Injuries in the own workforce,financial,operational,[73],[p73-0]
5,r05,Injuries to value chain workers,financial,operational,[73],[p73-1]
6,r06,Forced or child labour fines,regulatory,supply chain,[73],[p73-2]
7,r07,Insufficient market conditions,market,financial,[74],[p74-0]
8,r08,Corruption and bribery,regulatory,financial,[74],[p74-2]


## 8. Save the run → `merge/runs.py`

A merge run is saved as one JSON file in `runs/merge/`: the config (which names the extraction run it read), the
exact system prompt, the mentions as sent, the call (risks, usage or error), the problems the code fixed, and the
final records. The file is self-contained, so notebook 02b and the inspection below need nothing else. A run is
never overwritten, and a failed call is saved too: every attempt is kept.

In [8]:
from annual_report_reader.merge.runs import MERGE_RUNS_DIR, MergeRun, load_merge_run, save_merge_run

merge_run = MergeRun(
    run_id=merge_run_id,
    started_at=started_at,
    elapsed_s=round(elapsed, 1),
    config=CONFIG,
    system_prompt=prompt.format_messages(mentions="")[0].content,
    output_schema=MergedRisks.model_json_schema(),
    mentions=mentions,
    call=call,
    problems=problems,
    records=records,
)
merge_run_path = save_merge_run(merge_run, REPO_ROOT / MERGE_RUNS_DIR)
assert load_merge_run(merge_run_path) == merge_run  # the file loads back
print(f"Saved {merge_run_path.relative_to(REPO_ROOT)}")

Saved runs/merge/merge-20261112T012912Z-53ad.json


## 9. Inspect

Read back from the file just saved. Each record that was merged from several mentions is shown next to those
mentions, in the model's order (most informative first), to check two things by eye: that the mentions really are
the same risk, and that the description and mitigation keep what matters from all of them. The scores are notebook
02b's job.

In [9]:
import html

from IPython.display import HTML, display

from annual_report_reader.merge.mentions import Mention
from annual_report_reader.merge.schema import RiskRecord


def _column(heading: str, values: dict, keys: list[str]) -> str:
    """One labelled column: a heading, then one row per field."""
    rows = "".join(
        f"<tr><td style='vertical-align:top;padding-right:8px;opacity:.7'>{k}</td>"
        f"<td style='text-align:left'>{html.escape(str(values[k])).replace(chr(10), '<br>')}</td></tr>"
        for k in keys
    )
    table = f"<table style='font-size:12px;width:100%;table-layout:fixed'><col style='width:90px'><col>{rows}</table>"
    return f"<div style='flex:1;min-width:0'><b>{heading}</b>{table}</div>"


def show_merged(record: RiskRecord, mentions: list[Mention]) -> None:
    """The record next to the mentions it was built from."""
    by_id = {m.mention_id: m for m in mentions}
    locations = "; ".join(f"p.{loc.page}: {loc.section}" for loc in record.locations)
    cols = [
        _column(
            f"{record.id} (merged)",
            {**record.model_dump(), "locations": locations},
            ["title", "category", "locations", "description", "mitigation"],
        )
    ]
    cols += [
        _column(i, by_id[i].model_dump(), ["title", "category", "section", "description", "mitigation"])
        for i in record.mention_ids
    ]
    display(HTML(f"<div style='display:flex;gap:16px;margin:12px 0;text-align:left'>{''.join(cols)}</div>"))


loaded = load_merge_run(merge_run_path)
print(
    f"{loaded.run_id} (from {loaded.config.extraction_run}): {len(loaded.records)} records, {len(loaded.problems)} problems"
)
for record in loaded.records:
    if len(record.mention_ids) > 1:
        show_merged(record, loaded.mentions)

merge-20261112T012912Z-53ad (from extract-20261112T000123Z): 9 records, 0 problems


title,Cyber security risks
category,cyber
locations,p.51: Main risks - Cyber attacks; p.118: G1 – Entity-specific Cyber security; p.74: G1 Business Conduct - Financial risks or opportunities
description,"Vestas’ digitally connected wind turbines, infrastructure, technology, data and operations are exposed to evolving cyber attacks from geopolitical actors and cyber criminals. Successful attacks could disrupt energy production and business operations, cause outages or physical damage, expose intellectual property, create legal liability, erode partner trust and result in significant financial, economic and reputational losses."
mitigation,"Vestas is strengthening cybersecurity resilience through increased investment across R&D, service and digital infrastructure, open-source initiatives supporting cyber risk quantification, and customer-specific mitigation strategies. It is also developing cybersecurity services to help customers manage cyber risks and meet industry requirements and standards; no critical cybersecurity incidents took place during 2025."
title,Cyber attacks
category,cyber
section,Main risks - Cyber attacks
description,"Vestas’ digital and other critical assets are exposed to cyber attacks. A cyber attack may have dual implications for both Vestas as a company and for the societies that our energy solutions serve.Electricity producers rely on Vestas’ energy solutions for reliable and secure sustainable energy production, making these solutions critical infrastructure for societies around the world.Cyber-threat actors constantly evolve their methods and techniques to attack Vestas’ energy solutions, for the purpose of extortion or to destabilise energy generation. Vestas’ technology, data, and operations are a potential target for geopolitical actors and the cyber-criminal industry.Politically motivated cyber actors may seek to disrupt energy production. A cyber attack on Vestas may be designed to impact our customers’ operations, causing electricity outage and physical damage to assets.A successful cyber attack on Vestas’ energy solutions and our infrastructures might result in significant economic loss and reputational damage."
mitigation,"As a global manufacturer and key contributor to critical infrastructure, Vestas is required to continuously address evolving cyber risks, both to our company and to the customers we serve.We are developing cyber security services aimed at helping our customers navigate cyber risks effectively, fulfil industry requirements and standards, and uphold their responsibilities.During 2025, our cyber security efforts were effective and no critical cyber security incidents took place."
title,Cyber security risks (entity-specific)


title,Carbon taxes and tariffs
category,financial
locations,"p.85: Climate Change - Impacts, risks, and opportunities; p.71: E1 Climate Change - Financial risks or opportunities"
description,"High carbon taxes and tariffs on greenhouse-gas-intensive materials such as steel can increase the cost of producing wind turbines. If Vestas cannot distribute these costs along its value chain or transition to low-carbon alternatives quickly enough, its cost base may be adversely affected."
mitigation,None
title,Carbon taxes and tariffs
category,financial
section,"Climate Change - Impacts, risks, and opportunities"
description,"Our business outlook is influenced by political and legal developments. Political and market transition risks, such as carbon taxes, can financially impact Vestas. High carbon taxes and tariffs will increase the cost of GHG-intensive materials like steel, raising the overall cost of wind turbines. If we cannot distribute these costs along our value chain or transition to low-carbon alternatives fast enough, our cost base may be impacted."
mitigation,None
title,Carbon taxes and tariffs


## 10. Observations

_To fill in after reviewing the run._